# Observable Structure Function in LLM Data

**MVP demonstration** for the context-selection essays. Core claim: the Vereshchagin–Vitányi structure function is *observable* in real LLM behavior — different content types produce genuinely different compression curves, and respecting those shapes improves context selection decisions.

This notebook:
1. Takes a curated corpus of 10 content items (known, novel, mixed)
2. Approximates each item's compression curve by measuring response quality at multiple compression levels
3. Visualizes the curves — showing different shapes for different content types
4. Compares a threshold-based selector vs. a curve-aware selector on a fixed budget
5. Shows the curve-aware selector wins

In [ ]:
# Cell 1: Setup

import os
import random
import re
import time
import json
from dataclasses import dataclass, field
from pathlib import Path

import anthropic
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# ── Load API key ──
load_dotenv()  # reads .env from the working dir -- copy .env.example to .env and add your API key
api_key = os.environ.get("CLAUDE_API_KEY") or os.environ.get("ANTHROPIC_API_KEY")
assert api_key, "No API key found. Set CLAUDE_API_KEY in .env"

client = anthropic.Anthropic(api_key=api_key)

# ── Model selection ──
# Claude Haiku 4.5 — frozen snapshot, cheap, sufficient for the structure-function
# measurement. Any frozen decoder works; the theory is decoder-agnostic.
MODEL_ID = "claude-haiku-4-5-20251001"
test = client.messages.create(
    model=MODEL_ID,
    max_tokens=10,
    messages=[{"role": "user", "content": "ping"}],
)
print(f"Using {MODEL_ID}")

# ── Output paths ──
OUT_DIR = Path(".")  # figures + results are written next to this notebook; change if you want them elsewhere
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ──
TEMPERATURE = 0          # deterministic generation
COMPRESSION_LEVELS = [1.0, 0.75, 0.50, 0.25, 0.0]
RUNS_PER_LEVEL = 3       # average out sentence-sampling randomness
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Setup complete.")
print(f"Outputs will be written to: {OUT_DIR}")

In [ ]:
# Cell 2: Corpus

@dataclass
class ContentItem:
    name: str
    category: str                # "known", "novel", or "mixed"
    content: str                 # the actual text (~50-200 tokens)
    query: str                   # test question requiring info from content
    expected_facts: list[str]    # keywords/phrases a correct response should contain


corpus = [
    # ── KNOWN (model has these in training data) ──
    ContentItem(
        name="Sabine Hossenfelder",
        category="known",
        content=(
            "Sabine Hossenfelder is a German theoretical physicist and science communicator. "
            "She works at the Munich Center for Mathematical Philosophy. "
            "Her YouTube channel Science Without the Gobbledygook has over one million subscribers. "
            "She authored the book Lost in Math about the problems with beauty-driven physics research. "
            "She frequently criticizes string theory and the lack of empirical grounding in modern theoretical physics."
        ),
        query="Who is Sabine Hossenfelder and what is she known for?",
        expected_facts=["physicist", "youtube", "lost in math", "string theory", "gobbledygook"],
    ),
    ContentItem(
        name="Python decorators",
        category="known",
        content=(
            "In Python, a decorator is a function that takes another function as input and returns a modified version of it. "
            "Decorators use the @symbol syntax placed above a function definition. "
            "Common use cases include logging, access control, memoization, and timing. "
            "functools.wraps should be used inside custom decorators to preserve the original function's metadata. "
            "Decorators can also be implemented as classes with a __call__ method."
        ),
        query="Explain Python decorators including syntax and best practices.",
        expected_facts=["@", "function", "wraps", "logging", "__call__"],
    ),
    ContentItem(
        name="Marie Curie",
        category="known",
        content=(
            "Marie Curie was born Maria Sklodowska in Warsaw, Poland in 1867. "
            "She moved to Paris to study at the Sorbonne. "
            "She discovered the elements polonium and radium with her husband Pierre Curie. "
            "She was the first woman to win a Nobel Prize and the first person to win Nobel Prizes in two different sciences, physics and chemistry. "
            "She died in 1934 from aplastic anemia caused by prolonged radiation exposure."
        ),
        query="Summarize Marie Curie's key achievements and biographical details.",
        expected_facts=["polonium", "radium", "nobel", "sorbonne", "radiation"],
    ),
    ContentItem(
        name="Knitting basics",
        category="known",
        content=(
            "Knitting starts with casting on, which creates the initial row of stitches on the needle. "
            "The two fundamental stitches are knit and purl. "
            "Stockinette stitch alternates rows of knit and purl to create a smooth fabric on one side. "
            "Garter stitch uses only knit stitches and produces a ridged, stretchy fabric. "
            "Gauge swatches should be knitted before starting a project to ensure correct sizing."
        ),
        query="What are the basics of knitting including fundamental stitches and techniques?",
        expected_facts=["cast", "knit", "purl", "stockinette", "gauge"],
    ),

    # ── NOVEL (model has NOT seen these) ──
    ContentItem(
        name="Silvermoor game postmortem",
        category="novel",
        content=(
            "Silvermoor Studios released the puzzle-platformer Tidecaller on March 14 2026 after a two-year development cycle. "
            "The game shipped with 43 hand-crafted levels and a custom fluid physics engine called RipCurrent. "
            "Launch week sales hit 128,000 units across Steam and Itch combined. "
            "The team of seven developers spent six months on accessibility features including an audio-only mode. "
            "The biggest postmortem takeaway was that the in-house save system broke on Steam Deck and delayed the launch by three weeks."
        ),
        query="What were Silvermoor Studios' key launch results and postmortem findings for Tidecaller?",
        expected_facts=["silvermoor", "ripcurrent", "128", "steam deck", "accessibility"],
    ),
    ContentItem(
        name="Orenbrook dig",
        category="novel",
        content=(
            "The Orenbrook dig site in northern Scotland yielded 312 Bronze Age artefacts between May and September 2025. "
            "Lead archaeologist Dr. Hamish Drummond dated the primary burial chamber to approximately 1450 BCE using pottery typology. "
            "The most significant find was a bronze brooch inlaid with amber from the Baltic region, suggesting long-distance trade networks. "
            "The site showed unusual use of fired clay floor tiles, which is rare for the period in that region. "
            "The final report is scheduled for the Proceedings of the Society of Antiquaries of Scotland in mid-2026."
        ),
        query="Summarize the Orenbrook archaeological site findings including dating and significance.",
        expected_facts=["orenbrook", "drummond", "1450", "baltic", "fired clay"],
    ),
    ContentItem(
        name="Zephyr sprint review",
        category="novel",
        content=(
            "Meeting: Project Zephyr Sprint Review, April 3 2026. "
            "Attendees: Marta Vega (PM), Deshi Okafor (backend), Lin Xiaoli (ML), Ravi Subramanian (infra). "
            "Decision: migrate the embedding store from Pinecone to Qdrant by end of Q2. "
            "Ravi raised a concern about Qdrant's lack of managed backup in EU regions. "
            "Lin proposed a fallback: keep a read-only Pinecone mirror for 90 days post-migration."
        ),
        query="What were the key decisions and concerns from the Project Zephyr meeting?",
        expected_facts=["qdrant", "pinecone", "ravi", "backup", "90 day"],
    ),
    ContentItem(
        name="Kestrel Green cooperative",
        category="novel",
        content=(
            "The Kestrel Green cooperative in Bristol is a 34-unit housing project completed in October 2025. "
            "The building uses cross-laminated timber from regional spruce and achieves Passivhaus certification through triple-glazed windows and an MVHR ventilation system. "
            "Residents share a communal tool library, laundry, and a rooftop greenhouse run by a rotating three-person committee. "
            "Monthly service charges are 87 pounds per unit, deliberately kept low through shared maintenance labour. "
            "The project was funded via a community share offer that raised 1.2 million pounds in sixty days."
        ),
        query="Describe the Kestrel Green cooperative housing project in Bristol.",
        expected_facts=["kestrel", "cross-laminated", "passivhaus", "mvhr", "1.2 million"],
    ),

    # ── MIXED (known concepts, novel specifics) ──
    ContentItem(
        name="MerchantPath fraud GNN",
        category="mixed",
        content=(
            "The MerchantPath fraud detection system applies graph neural networks to a transaction graph where nodes are merchants and edges are money transfers. "
            "The system uses a standard GraphSAGE architecture with mean aggregation over 2-hop neighbourhoods. "
            "What is novel is the temporal edge weighting scheme called EchoScore, which discounts old transactions using a learned decay function. "
            "Node embeddings feed into a logistic classifier trained on 18,000 labelled fraud cases. "
            "The system flags merchant clusters rather than individual accounts, which reduces false positives by 43 percent compared to the previous rules-based approach."
        ),
        query="How does MerchantPath use graph neural networks to detect fraud?",
        expected_facts=["graphsage", "2-hop", "echoscore", "merchant cluster", "43"],
    ),
    ContentItem(
        name="Brambleweed Rust runtime",
        category="mixed",
        content=(
            "Brambleweed is an async Rust runtime designed for ARM Cortex-M microcontrollers with under 64KB of RAM. "
            "It uses cooperative task scheduling with a fixed-size executor pool, similar to embassy-rs but with a smaller footprint. "
            "The novel contribution is a static allocation scheme called SeedPool that pre-allocates task futures at compile time using proc macros. "
            "Interrupt handlers wake tasks via a lock-free ring buffer to avoid priority inversion. "
            "Initial benchmarks show 40 percent lower memory use than embassy-rs on a reference Cortex-M0 plus target."
        ),
        query="What is Brambleweed and how does it differ from embassy-rs?",
        expected_facts=["cortex-m", "seedpool", "proc macro", "ring buffer", "40"],
    ),
]

print(f"Corpus: {len(corpus)} items")
for item in corpus:
    print(f"  [{item.category:6s}] {item.name} — {len(item.expected_facts)} expected facts")

In [ ]:
# Cell 3: Measurement engine

def split_sentences(text: str) -> list[str]:
    """Split text into sentences. Simple but sufficient."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p.strip()]


def compress(text: str, level: float, rng: random.Random) -> str:
    """
    Compress text to `level` fraction (1.0 = full, 0.0 = nothing).
    Randomly samples sentences to hit target.
    """
    if level <= 0.0:
        return ""
    if level >= 1.0:
        return text
    
    sentences = split_sentences(text)
    n_keep = max(1, round(len(sentences) * level))
    kept = sorted(rng.sample(range(len(sentences)), n_keep))
    return " ".join(sentences[i] for i in kept)


def query_model(query: str, context: str) -> str:
    """Send query + context to the model. Returns response text."""
    if context:
        user_msg = f"Context:\n{context}\n\nQuestion: {query}"
    else:
        user_msg = f"Question: {query}"
    
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": user_msg}],
    )
    return response.content[0].text


def score_facts(response: str, expected_facts: list[str]) -> float:
    """Fraction of expected facts found in response (case-insensitive substring match)."""
    response_lower = response.lower()
    hits = sum(1 for fact in expected_facts if fact.lower() in response_lower)
    return hits / len(expected_facts)


def measure_curve(
    item: ContentItem,
    levels: list[float] = COMPRESSION_LEVELS,
    runs: int = RUNS_PER_LEVEL,
) -> dict:
    """Measure fact retention curve for a single content item."""
    results = {"name": item.name, "category": item.category, "curves": {}}
    
    for level in levels:
        scores = []
        for run in range(runs):
            rng = random.Random(SEED + run)  # reproducible per run
            compressed = compress(item.content, level, rng)
            response = query_model(item.query, compressed)
            score = score_facts(response, item.expected_facts)
            scores.append(score)
        results["curves"][level] = {
            "mean": np.mean(scores),
            "std": np.std(scores),
            "scores": scores,
        }
    return results


print("Measurement engine ready.")
print(f"  Compression levels: {COMPRESSION_LEVELS}")
print(f"  Runs per level: {RUNS_PER_LEVEL}")
print(f"  Total API calls: ~{len(corpus) * len(COMPRESSION_LEVELS) * RUNS_PER_LEVEL}")

In [ ]:
# Cell 4: Run measurements

all_results = []
total = len(corpus)
start_time = time.time()

for i, item in enumerate(corpus):
    print(f"[{i+1}/{total}] Measuring: {item.name} ({item.category})...")
    item_start = time.time()
    result = measure_curve(item)
    elapsed = time.time() - item_start
    
    # Print curve inline
    curve_str = "  "
    for level in COMPRESSION_LEVELS:
        mean = result["curves"][level]["mean"]
        curve_str += f"{int(level*100):>3d}%: {mean:.2f}  "
    print(f"{curve_str}  ({elapsed:.1f}s)")
    
    all_results.append(result)

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.0f}s ({total_time/60:.1f} min)")

In [ ]:
# Cell 5: Visualize curves

COLORS = {"known": "#2196F3", "novel": "#F44336", "mixed": "#9C27B0"}
STYLES = {"known": "-", "novel": "--", "mixed": "-."}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle("Structure Function: Fact Retention vs. Compression Level", fontsize=14, fontweight="bold")

for ax, category, title in zip(
    axes, ["known", "novel", "mixed"], ["Known Content", "Novel Content", "Mixed Content"]
):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Compression level (fraction of content kept)")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.grid(True, alpha=0.3)
    
    items = [r for r in all_results if r["category"] == category]
    for r in items:
        levels = sorted(r["curves"].keys())
        means = [r["curves"][l]["mean"] for l in levels]
        stds = [r["curves"][l]["std"] for l in levels]
        ax.errorbar(
            levels, means, yerr=stds,
            marker="o", markersize=5, capsize=3,
            color=COLORS[category], linestyle=STYLES[category],
            label=r["name"], alpha=0.8,
        )
    ax.legend(fontsize=8, loc="lower right")

axes[0].set_ylabel("Fact retention rate")
plt.tight_layout()
plt.savefig(OUT_DIR / "structure_function_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'structure_function_curves.png'}")

In [ ]:
# Cell 5b: Combined overlay plot

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title("All Structure Function Curves", fontsize=14, fontweight="bold")
ax.set_xlabel("Compression level (fraction of content kept)", fontsize=11)
ax.set_ylabel("Fact retention rate", fontsize=11)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

# Plot individual curves with low alpha
for r in all_results:
    cat = r["category"]
    levels = sorted(r["curves"].keys())
    means = [r["curves"][l]["mean"] for l in levels]
    ax.plot(levels, means, marker="o", markersize=4,
            color=COLORS[cat], linestyle=STYLES[cat],
            alpha=0.35, linewidth=1)

# Plot category averages with full alpha
for cat, label in [("known", "Known (avg)"), ("novel", "Novel (avg)"), ("mixed", "Mixed (avg)")]:
    items = [r for r in all_results if r["category"] == cat]
    levels = sorted(items[0]["curves"].keys())
    avg_means = [np.mean([r["curves"][l]["mean"] for r in items]) for l in levels]
    ax.plot(levels, avg_means, marker="s", markersize=7,
            color=COLORS[cat], linestyle="-", linewidth=2.5,
            label=label, alpha=0.95)

ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "structure_function_overlay.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'structure_function_overlay.png'}")

In [ ]:
# Cell 6: Selector comparison
#
# Scenario: 10 items, ~100 tokens each. Budget = 5.0 "item-equivalents"
# (can fit 5 items at full context, or all 10 at 50%, etc.)
# We use the measured curves to simulate two allocation strategies.

BUDGET = 5.0  # expressed as "equivalent full items"


def get_retention(result: dict, level: float) -> float:
    """Get mean fact retention at a given compression level (interpolates)."""
    if level in result["curves"]:
        return result["curves"][level]["mean"]
    levels = sorted(result["curves"].keys())
    for i in range(len(levels) - 1):
        if levels[i] <= level <= levels[i + 1]:
            t = (level - levels[i]) / (levels[i + 1] - levels[i])
            lo = result["curves"][levels[i]]["mean"]
            hi = result["curves"][levels[i + 1]]["mean"]
            return lo + t * (hi - lo)
    return 0.0


# ── Selector A: Threshold-based (naive) ──
# Uniform allocation: give every item the same fraction of context.
# At budget=5 with 10 items, each gets 50%.
def selector_threshold(results: list[dict], budget: float) -> list[tuple[str, float, float]]:
    """Uniform 50% allocation to all items within budget."""
    alloc_per_item = budget / len(results)
    return [(r["name"], alloc_per_item, get_retention(r, alloc_per_item)) for r in results]


# ── Selector B: Curve-aware (greedy marginal value) ──
# Key insight: known items barely degrade at 25%, so they're cheap to include.
# Novel items degrade steeply, so they need full context.
# Allocate by curve shape: flat → 25%, steep → 100%, moderate → 50%.
def selector_curve_aware(results: list[dict], budget: float) -> list[tuple[str, float, float]]:
    """Allocate based on measured curve steepness."""
    allocations = []
    for r in results:
        full_ret = get_retention(r, 1.0)
        low_ret = get_retention(r, 0.25)
        drop = full_ret - low_ret  # how much info is lost at 25%
        allocations.append({"result": r, "drop": drop})
    
    # Phase 1: Start everyone at 25% (minimum viable inclusion)
    plan = {}
    for info in allocations:
        name = info["result"]["name"]
        plan[name] = {"result": info["result"], "alloc": 0.25, "drop": info["drop"]}
    
    remaining = budget - 0.25 * len(allocations)  # budget left after minimum allocation
    
    # Phase 2: Upgrade steep-curve items, steepest first
    # Each upgrade from 0.25→1.0 costs 0.75 budget units
    by_steepness = sorted(plan.values(), key=lambda x: x["drop"], reverse=True)
    
    for item in by_steepness:
        name = item["result"]["name"]
        if item["drop"] > 0.3 and remaining >= 0.75:
            # Steep curve: upgrade to full context
            plan[name]["alloc"] = 1.0
            remaining -= 0.75
        elif item["drop"] > 0.15 and remaining >= 0.25:
            # Moderate: upgrade to 50%
            plan[name]["alloc"] = 0.50
            remaining -= 0.25
        # Flat curves stay at 25%
    
    # Build result
    selected = []
    for name, info in plan.items():
        ret = get_retention(info["result"], info["alloc"])
        selected.append((name, info["alloc"], ret))
    
    return selected


# ── Run both selectors ──
sel_a = selector_threshold(all_results, BUDGET)
sel_b = selector_curve_aware(all_results, BUDGET)

total_a = sum(r for _, _, r in sel_a)
total_b = sum(r for _, _, r in sel_b)
budget_a = sum(a for _, a, _ in sel_a)
budget_b = sum(a for _, a, _ in sel_b)

print("=" * 70)
print("SELECTOR A (Uniform): every item gets 50% compression")
print("=" * 70)
for name, alloc, ret in sel_a:
    print(f"  {name:28s}  alloc={alloc:.0%}  retention={ret:.0%}")
print(f"  Budget used: {budget_a:.2f} / {BUDGET}")
print(f"  Total fact retention: {total_a:.2f}")

print()
print("=" * 70)
print("SELECTOR B (Curve-aware): allocate by curve shape")
print("=" * 70)
for name, alloc, ret in sel_b:
    print(f"  {name:28s}  alloc={alloc:.0%}  retention={ret:.0%}")
print(f"  Budget used: {budget_b:.2f} / {BUDGET}")
print(f"  Total fact retention: {total_b:.2f}")

print()
print("=" * 70)
delta = total_b - total_a
pct = (delta / total_a * 100) if total_a > 0 else 0
if delta > 0:
    print(f"RESULT: Curve-aware wins by {delta:.2f} retained facts (+{pct:.0f}%)")
elif delta < 0:
    print(f"RESULT: Uniform wins by {-delta:.2f} — worth investigating.")
else:
    print("RESULT: Tie.")
print("=" * 70)

In [ ]:
# Cell 6b: Selector comparison visualization

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Selector Comparison: Threshold vs. Curve-Aware", fontsize=14, fontweight="bold")

# Left: allocation comparison
names_a = [n for n, _, _ in sel_a]
allocs_a = [a for _, a, _ in sel_a]

names_b = [n for n, _, _ in sel_b]
allocs_b = [a for _, a, _ in sel_b]

all_names = list(dict.fromkeys(names_a + names_b))
alloc_dict_a = dict(zip(names_a, allocs_a))
alloc_dict_b = dict(zip(names_b, allocs_b))

x = np.arange(len(all_names))
w = 0.35
bars_a = [alloc_dict_a.get(n, 0) for n in all_names]
bars_b = [alloc_dict_b.get(n, 0) for n in all_names]

ax1.barh(x - w/2, bars_a, w, label="Threshold", color="#78909C", alpha=0.8)
ax1.barh(x + w/2, bars_b, w, label="Curve-aware", color="#FF7043", alpha=0.8)
ax1.set_yticks(x)
ax1.set_yticklabels(all_names, fontsize=9)
ax1.set_xlabel("Budget allocation (fraction of full)")
ax1.set_title("Budget Allocation")
ax1.legend()
ax1.grid(True, axis="x", alpha=0.3)

ax2.bar(["Threshold", "Curve-aware"], [total_a, total_b],
        color=["#78909C", "#FF7043"], alpha=0.8, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Total retained facts (sum across items)")
ax2.set_title("Aggregate Fact Retention")
ax2.grid(True, axis="y", alpha=0.3)
for i, v in enumerate([total_a, total_b]):
    ax2.text(i, v + 0.05, f"{v:.2f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "selector_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'selector_comparison.png'}")

In [ ]:
# Cell 7: Results summary

print("=" * 70)
print("SUMMARY: Observable Structure Function in LLM Data")
print("=" * 70)

rows = []
for r in all_results:
    full = get_retention(r, 1.0)
    half = get_retention(r, 0.5)
    quarter = get_retention(r, 0.25)
    zero = get_retention(r, 0.0)
    drop = full - quarter
    rows.append({
        "Item": r["name"],
        "Category": r["category"],
        "100%": f"{full:.0%}",
        "50%": f"{half:.0%}",
        "25%": f"{quarter:.0%}",
        "0%": f"{zero:.0%}",
        "Drop (100→25)": f"{drop:.0%}",
    })

df = pd.DataFrame(rows)
print("\n" + df.to_string(index=False))

print("\n\nCategory averages (fact retention):")
for cat in ["known", "novel", "mixed"]:
    items = [r for r in all_results if r["category"] == cat]
    line = f"  {cat:6s}"
    for level in COMPRESSION_LEVELS:
        avg = np.mean([r["curves"][level]["mean"] for r in items])
        line += f"  @{int(level*100):>3d}%: {avg:.0%}"
    print(line)

print("\n")
print("KEY FINDINGS:")
print("-" * 70)

known_drop = np.mean([get_retention(r, 1.0) - get_retention(r, 0.25)
                      for r in all_results if r["category"] == "known"])
novel_drop = np.mean([get_retention(r, 1.0) - get_retention(r, 0.25)
                      for r in all_results if r["category"] == "novel"])
mixed_drop = np.mean([get_retention(r, 1.0) - get_retention(r, 0.25)
                      for r in all_results if r["category"] == "mixed"])
known_zero = np.mean([get_retention(r, 0.0) for r in all_results if r["category"] == "known"])
novel_zero = np.mean([get_retention(r, 0.0) for r in all_results if r["category"] == "novel"])

print(f"1. Known content: avg drop 100%→25% = {known_drop:.0%}")
print(f"   Known content at 0% (no context): {known_zero:.0%} — model reconstructs from weights alone.")
print()
print(f"2. Novel content: avg drop 100%→25% = {novel_drop:.0%}")
print(f"   Novel content at 0% (no context): {novel_zero:.0%} — model has nothing to reconstruct.")
print()
print(f"3. Mixed content: avg drop 100%→25% = {mixed_drop:.0%}")
print(f"   Known parts survive compression; novel specifics degrade.")
print()
print(f"4. Selector comparison: Curve-aware = {total_b:.2f}, Uniform = {total_a:.2f}")
if total_b > total_a:
    print(f"   Curve-aware wins by {total_b - total_a:.2f} ({(total_b - total_a)/total_a:.0%})")

In [ ]:
# Cell 8: Save raw data for reproducibility

def make_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

serializable_results = []
for r in all_results:
    sr = {"name": r["name"], "category": r["category"], "curves": {}}
    for level, data in r["curves"].items():
        sr["curves"][str(level)] = {
            "mean": make_serializable(data["mean"]),
            "std": make_serializable(data["std"]),
            "scores": [make_serializable(s) for s in data["scores"]],
        }
    serializable_results.append(sr)

output_path = OUT_DIR / "mvp_raw_results.json"
with open(output_path, "w") as f:
    json.dump({
        "model": MODEL_ID,
        "temperature": TEMPERATURE,
        "seed": SEED,
        "compression_levels": COMPRESSION_LEVELS,
        "runs_per_level": RUNS_PER_LEVEL,
        "results": serializable_results,
    }, f, indent=2)

print(f"Raw results saved to {output_path}")
print(f"Notebook complete. All {len(all_results)} items measured, visualized, and compared.")